In [7]:
import numpy as np
import pandas as pd
from itables import show

df = pd.read_parquet("../data/portfolio_priced.parquet")
df.shape

(668897, 18)

In [2]:
df.head()

,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,ClaimAmount,age_group,bm_group,freq_pred,sev_pred,prime_pure
66,139.0,1,0.75,F,7,1,61,50,B12,'Regular',27000,R11,303.00,"(55, 65]","(49, 60]",0.056376,1756.351908,127.114796
93,190.0,1,0.14,B,12,5,50,60,B12,'Diesel',56,R25,1981.84,"(45, 55]","(49, 60]",0.064100,1806.830859,143.917067
199,414.0,1,0.14,E,4,0,36,85,B12,'Regular',4792,R11,1456.55,"(35, 45]","(80, 100]",0.163116,1893.518779,336.962537
205,424.0,2,0.62,F,10,0,51,100,B12,'Regular',27000,R11,10834.00,"(45, 55]","(80, 100]",0.187014,1978.799810,398.162354
223,463.0,1,0.31,A,5,0,45,50,B12,'Regular',12,R73,3986.67,"(35, 45]","(49, 60]",0.055909,1728.961235,124.763448


In [3]:
df["decile"] = pd.qcut(df["prime_pure"], 10, labels=False, duplicates="drop")

resume = df.groupby("decile").apply(lambda g: pd.Series({
    "prime_pure_moy": np.average(g["prime_pure"], weights=g["Exposure"]),
    "cout_reel_moy": g["ClaimAmount"].sum() / g["Exposure"].sum(),
    "part_du_cout_total": g["ClaimAmount"].sum() / df["ClaimAmount"].sum(),
}))

resume

,prime_pure_moy,cout_reel_moy,part_du_cout_total
decile,,,
0,118.113133,93.961784,0.162963
1,127.114796,107.334493,0.082685
2,143.181937,122.327345,0.243556
3,219.825918,217.197209,0.127817
4,272.580133,269.044304,0.094870
5,396.563497,612.543747,0.288109


In [4]:
print(df["prime_pure"].nunique())

36


In [8]:
cells = (df.groupby(["age_group", "bm_group"])
           .apply(lambda g: pd.Series({
               "prime_pure": g["prime_pure"].iloc[0],
               "exposition": g["Exposure"].sum(),
               "cout": g["ClaimAmount"].sum(),
           }))
           .reset_index()
           .sort_values("prime_pure"))

cells["cout_reel"] = cells["cout"] / cells["exposition"]
cells["expo_cum"] = cells["exposition"].cumsum() / cells["exposition"].sum()
cells["groupe"] = np.ceil(cells["expo_cum"] * 10).clip(upper=10).astype(int)

show(cells)

In [9]:
resume = cells.groupby("groupe").apply(lambda c: pd.Series({
    "nb_cellules": len(c),
    "part_exposition": c["exposition"].sum() / cells["exposition"].sum(),
    "prime_pure_moy": np.average(c["prime_pure"], weights=c["exposition"]),
    "cout_reel_moy": c["cout"].sum() / c["exposition"].sum(),
    "part_du_cout_total": c["cout"].sum() / cells["cout"].sum(),
}))
resume

,nb_cellules,part_exposition,prime_pure_moy,cout_reel_moy,part_du_cout_total
groupe,,,,,
1,1.0,0.094998,104.209938,81.979798,0.046005
3,1.0,0.198604,124.763448,99.693128,0.116959
5,2.0,0.132800,127.271413,107.759463,0.084534
6,1.0,0.113298,141.901177,123.799095,0.082855
8,1.0,0.221361,143.917067,121.481074,0.158851
9,4.0,0.138742,233.340704,236.099957,0.193501
10,26.0,0.100197,372.989165,536.081349,0.317295
